In [ ]:
import os
import sys

# 1. MONTAGGIO DRIVE E SETUP PERCORSI
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    print(f"Directory impostata su: {os.getcwd()}")
except ImportError:
    # Fallback per esecuzione in locale
    DRIVE_ROOT = '.'
    print("Esecuzione in locale")

# 2. INSTALLAZIONI DI SISTEMA (Usiamo ! bash commands per massima stabilità)
print("\n--- Inizio Installazione Transformers e Qwen ---")
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install qwen-vl-utils accelerate

!pip install "bitsandbytes>=0.46.1"

print("\n--- Utility Pipeline e Fix OpenCV ---")
!pip install "opencv-python==4.10.0.84" "opencv-contrib-python==4.10.0.84" "opencv-python-headless==4.10.0.84"

print("\n=======================================================")
print("INSTALLAZIONE COMPLETATA. RIAVVIO IL KERNEL AUTOMATICAMENTE.")
print("Attendi 5 secondi, poi esegui la cella con il tuo script Python.")
print("=======================================================")

# 3. RIAVVIO AUTOMATICO DEL RUNTIME
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


In [ ]:
import sys
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    print(f"Directory impostata su: {os.getcwd()}")
except ImportError:
    # Fallback per esecuzione in locale
    DRIVE_ROOT = '.'
    print("Esecuzione in locale")

# Cambia la directory di lavoro in modo che i path relativi (es. configs/, saved_models/) funzionino
os.chdir(DRIVE_ROOT)
print(f"Directory di lavoro ripristinata su: {os.getcwd()}")

In [ ]:
import os
import json
import glob
import torch
import gc
import warnings
import numpy as np
from PIL import Image, ImageDraw
import tempfile
import shutil
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

# ==========================================
# CONFIGURAZIONE E PARAMETRI
# ==========================================
PATH_DATA_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT'
PATH_VIDEOS_CROP_DECODE = os.path.join(PATH_DATA_ROOT, "Videos_crop_decode")
PATH_OBJ_DET_FINISHED = os.path.join(PATH_DATA_ROOT, "video_general_obj_det_finished")
PATH_OUTPUT_DATASET = os.path.join(PATH_DATA_ROOT, "action_recognition_finished")

AZIONI_CONSENTITE = [
    "lift", "carry", "push", "pull", "pass", "lean_on", "touch", "hit", "play(instrument)", "grab",
    "release", "press", "use", "throw", "clean", "knock", "squeeze", "cut", "open", "close", "watch",
    "hold", "speak_to", "ride", "hug", "hold_hand_of", "bite", "caress", "pat", "wave", "point_to",
    "chase", "feed", "kiss", "kick", "smell", "wave_hand_to", "lick", "drive", "shout_at", "get_on", "get_off",
    "shake_hand_with"
]

WINDOW_SIZE = 30
STRIDE = 15
FRAMES_TO_SAMPLE = 15

EFFECTIVE_FPS = int(FRAMES_TO_SAMPLE / (WINDOW_SIZE / 30.0))

os.makedirs(PATH_OUTPUT_DATASET, exist_ok=True)

# ==========================================
# INIZIALIZZAZIONE MODELLO (Ottimizzato per T4)
# ==========================================
model_id = "Qwen/Qwen3-VL-8B-Instruct"

print(f"Caricamento del modello {model_id} in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"
)
processor = AutoProcessor.from_pretrained(model_id)
model.eval()

# ==========================================
# FUNZIONI UTILI PER VISUAL PROMPTING
# ==========================================
def draw_bboxes_on_frames(frame_paths, person_boxes, objects_boxes, temp_dir):
    """
    Disegna una bbox rossa sul target e blu sugli oggetti, salvando frame temporanei.
    Usa frame_paths e le liste di coordinate corrispondenti per quel frame.
    Coordinate in input previste come normalizzate [0, 1].
    """
    drawn_paths = []

    for i, path in enumerate(frame_paths):
        img = Image.open(path).convert("RGB")
        draw = ImageDraw.Draw(img)
        w, h = img.size

        # Disegna target (Rosso, molto evidente)
        if i < len(person_boxes) and person_boxes[i] is not None:
            px1, py1, px2, py2 = person_boxes[i]
            draw.rectangle([px1*w, py1*h, px2*w, py2*h], outline="red", width=5)

        # Disegna oggetti (Blu, meno spessi)
        for obj_boxes in objects_boxes:
            if i < len(obj_boxes) and obj_boxes[i] is not None:
                ox1, oy1, ox2, oy2 = obj_boxes[i]
                draw.rectangle([ox1*w, oy1*h, ox2*w, oy2*h], outline="blue", width=3)

        temp_path = os.path.join(temp_dir, f"frame_{i:03d}.jpg")
        img.save(temp_path)
        drawn_paths.append(temp_path)

    return drawn_paths

def process_window(drawn_frame_paths, action_list):
    """Calcola l'azione sfruttando la Text Generation vincolata."""

    actions_str = ", ".join(action_list)

    # Prompt chiaro, che si affida al visual prompting (box rosso)
    prompt_text = (
        "You are an expert action recognition system. Analyze the provided video sequence.\n"
        "The primary actor you must analyze is enclosed in a thick RED bounding box. "
        "Other interactable objects are in BLUE bounding boxes.\n"
        "Identify the single most prominent action being performed by the person in the RED box.\n"
        f"You MUST choose exactly ONE action from this exact list: [{actions_str}].\n\n"
        "Respond with ONLY the exact name of the action from the list. Do not add any punctuation or other text."
    )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": drawn_frame_paths,
                    "max_pixels": 768 * 768,
                    "fps": EFFECTIVE_FPS,
                },
                {
                    "type": "text",
                    "text": prompt_text
                }
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        return_video_kwargs=True,
        return_video_metadata=True
    )

    if video_inputs is not None:
        video_inputs, video_metadatas = zip(*video_inputs)
        video_inputs = list(video_inputs)
        video_metadatas = list(video_metadatas)
    else:
        video_metadatas = None

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        video_metadata=video_metadatas,
        padding=True,
        return_tensors="pt",
        **video_kwargs
    ).to("cuda")

    # Usiamo la generazione standard, limitando i token per evitare allucinazioni
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False # Greedy decoding per massima determinanza e precisione
        )

        # Isola solo l'output generato
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

    del inputs, generated_ids
    torch.cuda.empty_cache()

    # Validazione base dell'output
    predicted_action = output_text if output_text in action_list else "unknown"
    return predicted_action

# ==========================================
# LOOP PRINCIPALE DI ELABORAZIONE
# ==========================================
video_folders = glob.glob(os.path.join(PATH_OBJ_DET_FINISHED, "*"))

for video_folder in video_folders:
    video_name = os.path.basename(video_folder)
    print(f"\n[+] Elaborazione Video: {video_name}")

    json_files = glob.glob(os.path.join(video_folder, "*.json"))
    if not json_files:
        print(f"  [Skip] Nessun JSON trovato per {video_name}")
        continue

    with open(json_files[0], 'r') as f:
        tracking_data = json.load(f)

    frames_dir = os.path.join(PATH_VIDEOS_CROP_DECODE, video_name)
    all_frames = sorted(glob.glob(os.path.join(frames_dir, "*.jpg")))
    total_frames = len(all_frames)

    if total_frames == 0:
        continue

    detected_objects = tracking_data.get("detected_objects", {})
    people_ids = [k for k in detected_objects.keys() if k.startswith("person_")]

    for person_id in people_ids:
        print(f"  Analisi azioni per il soggetto: {person_id}")
        person_bboxes = detected_objects[person_id].get("bbox", [])
        person_action_log = {}

        # Sostituito range semplice con iterazione a STRIDE (finestre sovrapposte)
        for start_idx in range(0, total_frames - WINDOW_SIZE + 1, STRIDE):
            end_idx = start_idx + WINDOW_SIZE

            window_frames = all_frames[start_idx:end_idx]
            sample_indices = np.linspace(0, len(window_frames) - 1, min(FRAMES_TO_SAMPLE, len(window_frames)), dtype=int)

            sampled_frames_paths = [window_frames[i] for i in sample_indices]

            # Estraiamo le bbox specifiche per i frame campionati
            sampled_person_bboxes = [
                person_bboxes[start_idx + i] if (start_idx + i) < len(person_bboxes) else None
                for i in sample_indices
            ]

            # Se la persona non è presente nella maggior parte della finestra, skippiamo
            if sampled_person_bboxes.count(None) > (FRAMES_TO_SAMPLE / 2):
                continue

            sampled_objects_bboxes = []
            for obj_key, obj_data in detected_objects.items():
                if obj_key.startswith("person_"): continue # Ignora altre persone per non confondere il modello

                obj_boxes = obj_data.get("bbox", [])
                s_obj_boxes = [
                    obj_boxes[start_idx + i] if (start_idx + i) < len(obj_boxes) else None
                    for i in sample_indices
                ]

                # Aggiungi solo se l'oggetto compare almeno in un frame campionato
                if any(box is not None for box in s_obj_boxes):
                    sampled_objects_bboxes.append(s_obj_boxes)

            # Creiamo una cartella temporanea per salvare i frame con i box disegnati
            temp_dir = tempfile.mkdtemp()
            try:
                drawn_paths = draw_bboxes_on_frames(
                    sampled_frames_paths,
                    sampled_person_bboxes,
                    sampled_objects_bboxes,
                    temp_dir
                )

                # Invocazione del VLM
                predicted_action = process_window(
                    drawn_frame_paths=drawn_paths,
                    action_list=AZIONI_CONSENTITE
                )

                timestamp_key = f"frames_{start_idx}_to_{end_idx}"
                person_action_log[timestamp_key] = {"action": predicted_action}
                print(f"    -> Finestra {start_idx}-{end_idx}: {predicted_action}")

            finally:
                # Pulizia sicura dei frame temporanei
                shutil.rmtree(temp_dir, ignore_errors=True)

        output_filename = f"{video_name}_{person_id}_actions.json"
        output_file_path = os.path.join(PATH_OUTPUT_DATASET, output_filename)

        with open(output_file_path, 'w', encoding='utf-8') as out_f:
            json.dump(person_action_log, out_f, indent=4, ensure_ascii=False)

        print(f"  Salvataggio completato: {output_file_path}")
        gc.collect()

print("\n[+] Pipeline completata con successo su tutti i video.")